In [22]:
#!pip freeze > requirements.txt

In [23]:
#Musical Machine Orchestration
#By Gissel Velarde
#Update 18.6.2025
#28.5.2025
import numpy as np
import os
import pandas as pd
import sys
sys.path.append('amos') 
from midi2df2midi import midi_to_dataframe, save_midi_from_df

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import time 

In [25]:
"""
Code generated with https://platform.openai.com/
Model gpt-4.1, text.format: text, temp: 1.00, tokens: 2048, top_p: 1.00, store: true
Prompt by Gissel Velarde
Prompt:
the first 4 columns of a an numpy array with 8 columns always appear in quaterna. Write a function that learns the quaterna, given that column 3 is used as a label for a machine learning model. Then, once the model predicts the label for column 4, fill the corresponding values for columns 0, 1 and 2.

MODIFIED: Enhanced to preserve multiple track-channel combinations per program
"""
#import numpy as np

def learn_quaterna_mapping(array):
    """
    Learns the mapping from label in column 3 to columns 0, 1, 2.
    Enhanced to preserve ALL track-channel combinations for each program.
    Returns a dictionary: label_val -> list of [col0, col1, col2]
    """
    mapping = {}
    for row in array:
        label = row[3]  # program number
        track_info = row[:3].tolist()  # [track_num, track_name, channel]
        
        if label not in mapping:
            mapping[label] = []
        
        # Only add if this specific combination isn't already present
        if track_info not in mapping[label]:
            mapping[label].append(track_info)
    
    return mapping

In [26]:
"""
Code generated with https://platform.openai.com/
Model gpt-4.1, text.format: text, temp: 1.00, tokens: 2048, top_p: 1.00, store: true
Prompt by Gissel Velarde
Prompt:
the first 4 columns of a an numpy array with 8 columns always appear in quaterna. Write a function that learns the quaterna, given that column 3 is used as a label for a machine learning model. Then, once the model predicts the label for column 4, fill the corresponding values for columns 0, 1 and 2.

MODIFIED: Enhanced to distribute notes across multiple instruments
"""
def fill_quaterna_columns(predicted_labels, quaterna_mapping):
    """
    Given a list/array of predicted labels, use the mapping to reconstruct cols 0, 1, 2.
    For programs with multiple instruments, distributes notes in round-robin fashion.
    Returns an array of shape (N, 3) where N is the number of predicted labels.
    """
    filled = []
    # Keep track of which instrument to use next for each program (round-robin)
    instrument_counters = {}
    
    for label in predicted_labels:
        if label in quaterna_mapping:
            available_instruments = quaterna_mapping[label]
            
            if len(available_instruments) == 1:
                # Single instrument - use it directly
                filled.append(available_instruments[0])
            else:
                # Multiple instruments - distribute in round-robin fashion
                if label not in instrument_counters:
                    instrument_counters[label] = 0
                
                # Select the next instrument in rotation
                selected_instrument = available_instruments[instrument_counters[label]]
                filled.append(selected_instrument)
                
                # Move to next instrument for this program
                instrument_counters[label] = (instrument_counters[label] + 1) % len(available_instruments)
        else:
            # handle unknown labels (e.g., with np.nan)
            filled.append([np.nan, np.nan, np.nan])
    
    return np.array(filled)

In [27]:
#By Gissel Velarde
#Update 18.6.2025
#28.5.2025
def split_and_encode(X, y, test_size=0.2, random_state=42):
    if test_size>0:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    elif test_size==0:
        X_train = X
        y_train = y
        X_test, y_test  = 0, 0 #We will no use X_test, y_test for inference
    le = LabelEncoder()
    le.fit(y_train)  # Fit only on train 
    print("y_train, test size:",test_size,", labels:", np.unique(y_train))
    y_train = le.transform(y_train)  # Will be 0,1,...,N-1
    if test_size>0:
        y_test = le.transform(y_test)
    return X_train, X_test, y_train, y_test, le

In [28]:
#By Gissel Velarde
#Update 18.6.2025
#28.5.2025
def clf_predict(X2,le,model,mapping):
    y_pred = model.predict(X2)
    print("Predictions ",np.unique(y_pred))
    # inverse transform to obtained the original labels:
    y_pred_orig = le.inverse_transform(y_pred)
    # Fill columns 0, 1, 2 using the mapping
    new_cols = fill_quaterna_columns(y_pred_orig, mapping)
    print("Predictions map",np.unique(y_pred_orig))
    nmat = np.concatenate((new_cols, y_pred_orig.reshape(-1, 1),X2), axis=1)
    return nmat

In [29]:
# By G. Velarde from
#16.6.2025
def ml_exp(filein, fileout):
    #Adapted from:
    # https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html
    # Authors: The scikit-learn developers
    # SPDX-License-Identifier: BSD-3-Clause
    names = [
        "Nearest_Neighbors",
        "Linear_SVM",
        "Decision_Tree",
        "Random_Forest",
        "Neural_Net",
        "AdaBoost",
        "Naive_Bayes",
        "XGBoost",
    ]

    classifiers = [
        KNeighborsClassifier(1),
        SVC(kernel="linear"),
        DecisionTreeClassifier(),
        RandomForestClassifier(),
        MLPClassifier(),
        AdaBoostClassifier(),
        GaussianNB(),
        XGBClassifier(),
    ]
    # midi to dataframe
    dfnmat = midi_to_dataframe(filein)
    # sort by onset, duration, track number
    dfnmat = dfnmat.sort_values(
        ['onset in quarter notes','duration in quarter notes', 'track number'],
        ascending=[True, True, True]
    )
    # convert df to nmat array
    nmat = dfnmat.to_numpy()  
    # Learn mapping from col3 /programs to cols 0-2, track, track name, channel
    mapping = learn_quaterna_mapping(nmat)
    print("mapping", mapping)
    X = nmat[:, 4:8]  # onset, duration, pitch, velocity
    y = nmat[:, 3]    # tracks
    print("Labels", np.unique(y))
    print("Number of events in ",filein,":", X.shape[0])
    print("last onset at ", X[X.shape[0]-1, 0])
    rng = np.random.RandomState(2)
    ### The outfile
    dfnmat = midi_to_dataframe(fileout) 
    #sort by onset, duration, track number
    dfnmat=dfnmat.sort_values(['onset in quarter notes','duration in quarter notes', 'track number'],ascending=[True, True, True])
    #convert df to nmat array
    nmat2=dfnmat.to_numpy()
    X2 = nmat2[:,4:8] #onset, duration, pitch, velocity
    print("Number of events in",fileout,":",X2.shape[0])
    print("last onset at ",X2[X2.shape[0]-1,0])
    #Partion the dataset
    X_train, X_test, y_train, y_test,le = split_and_encode(X, y, test_size=0.2, random_state=42) #For evaluation
    X_train_f, X_test_f, y_train_f, y_test_f,le_f = split_and_encode(X, y, test_size=0, random_state=42) #For orchestration
    # iterate over classifiers
    for name, clf in zip(names, classifiers):
        start = time.time()
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)
        end = time.time()
        print("---------",name)
        print("Train Time (sec) :",f"{end - start:.4f}")
        print("Score on Test (20%): ", f"{score:.4f}") 
        #_= ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test_encoded)
        #Predict orchestration 
        #Retrain with the full input file
        clf.fit(X_train_f, y_train_f)
        data=clf_predict(X2,le_f,clf,mapping)
        #convert nmat array to df
        #https://www.geeksforgeeks.org/convert-numpy-array-to-dataframe/
        #Specifying Column Names 				pitch	velocity
        dfdata = pd.DataFrame(data, columns=['track number','track name', 'channel', 'program','onset in quarter notes','duration in quarter notes','pitch','velocity'])
        
        # Fix data types to ensure mido compatibility
        dfdata['track number'] = dfdata['track number'].astype(int)
        dfdata['channel'] = dfdata['channel'].astype(int)
        dfdata['program'] = dfdata['program'].astype(int)
        dfdata['pitch'] = dfdata['pitch'].astype(int)
        dfdata['velocity'] = dfdata['velocity'].astype(int)
        dfdata['onset in quarter notes'] = dfdata['onset in quarter notes'].astype(float)
        dfdata['duration in quarter notes'] = dfdata['duration in quarter notes'].astype(float)
        dfdata['track name'] = dfdata['track name'].astype(str)
        
        extensions = name + ".mid"
        filename = fileout.replace(".mid", extensions)
        print('Orchestration:',filename)
        save_midi_from_df(dfdata, filename)

In [30]:
#Usage
#ml_exp(filein,fileout)
#filein corresponds to the file used to learn the orchestration style and instrument map
#fileout is the file used to transfer the learned orchestration style.
#ml_exp will save a new MIDI files for each ML model
ml_exp('midis/sugar-plum-fairy_orch.mid','midis/fur-elise.mid')

mapping {45: [[12, 'Violoncello', 0], [13, 'Contrabass', 1], [9, 'Violin I', 13], [10, 'Violin II', 14], [11, 'Viola', 15]], 8: [[8, 'Celesta', 0]], 71: [[5, 'Bass Clarinet in Bb', 2], [4, '2 Clarinets in A', 6], [4, '2 Clarinets in A', 7]], 69: [[3, 'English Horn', 5]], 70: [[6, '2 Bassoons', 8], [6, '2 Bassoons', 10]], 73: [[1, '3 Flutes', 1], [1, '3 Flutes', 2], [1, '3 Flutes', 3]], 68: [[2, '2 Oboes', 4]], 60: [[7, '4 Horns in F', 11], [7, '4 Horns in F', 12]], 48: [[13, 'Contrabass', 1], [10, 'Violin II', 14], [11, 'Viola', 15], [12, 'Violoncello', 0], [9, 'Violin I', 13]]}
Labels [8 45 48 60 68 69 70 71 73]
Number of events in  midis/sugar-plum-fairy_orch.mid : 1825
last onset at  103.5
Number of events in midis/fur-elise.mid : 1020
last onset at  186.5
y_train, test size: 0.2 , labels: [8 45 48 60 68 69 70 71 73]
y_train, test size: 0 , labels: [8 45 48 60 68 69 70 71 73]
--------- Nearest_Neighbors
Train Time (sec) : 0.0041
Score on Test (20%):  0.8932
Predictions  [1 2 3 6 7 8

c:\Repositories\AMO\AutomaticMusicOrchestration\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


--------- Neural_Net
Train Time (sec) : 0.5876
Score on Test (20%):  0.7863


c:\Repositories\AMO\AutomaticMusicOrchestration\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Predictions  [1 2 5 6 7]
Predictions map [45 48 69 70 71]
Orchestration: midis/fur-eliseNeural_Net.mid
--------- AdaBoost
Train Time (sec) : 0.1740
Score on Test (20%):  0.7699
Predictions  [0 1 2 5 6 7 8]
Predictions map [8 45 48 69 70 71 73]
Orchestration: midis/fur-eliseAdaBoost.mid
--------- AdaBoost
Train Time (sec) : 0.1740
Score on Test (20%):  0.7699
Predictions  [0 1 2 5 6 7 8]
Predictions map [8 45 48 69 70 71 73]
Orchestration: midis/fur-eliseAdaBoost.mid
--------- Naive_Bayes
Train Time (sec) : 0.0027
Score on Test (20%):  0.7726
Predictions  [0 2 4 6 7 8]
Predictions map [8 48 68 70 71 73]
Orchestration: midis/fur-eliseNaive_Bayes.mid
--------- Naive_Bayes
Train Time (sec) : 0.0027
Score on Test (20%):  0.7726
Predictions  [0 2 4 6 7 8]
Predictions map [8 48 68 70 71 73]
Orchestration: midis/fur-eliseNaive_Bayes.mid
--------- XGBoost
Train Time (sec) : 0.8708
Score on Test (20%):  0.9397
--------- XGBoost
Train Time (sec) : 0.8708
Score on Test (20%):  0.9397
Predictions  

In [31]:
# Test the improved multi-instrument mapping
print("=== Testing Multi-Instrument Mapping ===")

# Load the sugar plum fairy and check the new mapping
dfnmat = midi_to_dataframe('midis/sugar-plum-fairy_orch.mid')
nmat = dfnmat.to_numpy()
mapping = learn_quaterna_mapping(nmat)

print("Enhanced mapping with multiple instruments per program:")
for program, instruments in mapping.items():
    print(f"Program {program}: {len(instruments)} instrument(s)")
    for i, inst in enumerate(instruments):
        print(f"  {i+1}. Track {inst[0]}: {inst[1]} (Ch {inst[2]})")
    print()

# Test distribution for a program with multiple instruments
if 73 in mapping:  # Flutes program
    print("Testing note distribution for 3 Flutes (program 73):")
    # Simulate 9 notes being distributed across 3 flutes
    test_labels = [73] * 9
    result = fill_quaterna_columns(test_labels, mapping)
    
    print("Distribution pattern:")
    for i, inst_info in enumerate(result):
        track, name, channel = inst_info
        print(f"  Note {i+1}: Track {track} {name} (Ch {channel})")

=== Testing Multi-Instrument Mapping ===
Enhanced mapping with multiple instruments per program:
Program 73: 3 instrument(s)
  1. Track 1: 3 Flutes (Ch 1)
  2. Track 1: 3 Flutes (Ch 2)
  3. Track 1: 3 Flutes (Ch 3)

Program 68: 1 instrument(s)
  1. Track 2: 2 Oboes (Ch 4)

Program 69: 1 instrument(s)
  1. Track 3: English Horn (Ch 5)

Program 71: 3 instrument(s)
  1. Track 4: 2 Clarinets in A (Ch 6)
  2. Track 4: 2 Clarinets in A (Ch 7)
  3. Track 5: Bass Clarinet in Bb (Ch 2)

Program 70: 2 instrument(s)
  1. Track 6: 2 Bassoons (Ch 8)
  2. Track 6: 2 Bassoons (Ch 10)

Program 60: 2 instrument(s)
  1. Track 7: 4 Horns in F (Ch 11)
  2. Track 7: 4 Horns in F (Ch 12)

Program 8: 1 instrument(s)
  1. Track 8: Celesta (Ch 0)

Program 45: 5 instrument(s)
  1. Track 9: Violin I (Ch 13)
  2. Track 10: Violin II (Ch 14)
  3. Track 11: Viola (Ch 15)
  4. Track 12: Violoncello (Ch 0)
  5. Track 13: Contrabass (Ch 1)

Program 48: 5 instrument(s)
  1. Track 9: Violin I (Ch 13)
  2. Track 10: Viol

In [32]:
# Test the improved system with just one ML model
print("=== Testing Enhanced Multi-Instrument Orchestration ===")

# Quick test with just XGBoost (the best performer)
from sklearn.ensemble import RandomForestClassifier

filein = 'midis/sugar-plum-fairy_orch.mid'
fileout = 'midis/fur-elise.mid'

# Load and process data
dfnmat = midi_to_dataframe(filein)
dfnmat = dfnmat.sort_values(['onset in quarter notes','duration in quarter notes', 'track number'], ascending=[True, True, True])
nmat = dfnmat.to_numpy()
mapping = learn_quaterna_mapping(nmat)

X = nmat[:, 4:8]  # onset, duration, pitch, velocity
y = nmat[:, 3]    # programs

dfnmat2 = midi_to_dataframe(fileout)
dfnmat2 = dfnmat2.sort_values(['onset in quarter notes','duration in quarter notes', 'track number'], ascending=[True, True, True])
nmat2 = dfnmat2.to_numpy()
X2 = nmat2[:,4:8]

# Train model
X_train_f, X_test_f, y_train_f, y_test_f, le_f = split_and_encode(X, y, test_size=0, random_state=42)
clf = RandomForestClassifier()
clf.fit(X_train_f, y_train_f)

# Predict and create orchestration
data = clf_predict(X2, le_f, clf, mapping)
dfdata = pd.DataFrame(data, columns=['track number','track name', 'channel', 'program','onset in quarter notes','duration in quarter notes','pitch','velocity'])

# Fix data types
dfdata['track number'] = dfdata['track number'].astype(int)
dfdata['channel'] = dfdata['channel'].astype(int)
dfdata['program'] = dfdata['program'].astype(int)
dfdata['pitch'] = dfdata['pitch'].astype(int)
dfdata['velocity'] = dfdata['velocity'].astype(int)
dfdata['onset in quarter notes'] = dfdata['onset in quarter notes'].astype(float)
dfdata['duration in quarter notes'] = dfdata['duration in quarter notes'].astype(float)
dfdata['track name'] = dfdata['track name'].astype(str)

# Save result
save_midi_from_df(dfdata, 'midis/fur-elise_Enhanced_Test.mid')

# Analyze the result
print("\nResult Analysis:")
result_tracks = dfdata.groupby(['track number', 'track name', 'channel']).size().reset_index(name='note_count')
print(f"Total track-channel combinations: {len(result_tracks)}")
print("\nTrack structure in enhanced orchestration:")
for _, row in result_tracks.iterrows():
    track_num = row['track number']
    track_name = row['track name']
    channel = row['channel']
    note_count = row['note_count']
    print(f"  Track {track_num}: {track_name} (Ch {channel}) - {note_count} notes")

=== Testing Enhanced Multi-Instrument Orchestration ===
y_train, test size: 0 , labels: [8 45 48 60 68 69 70 71 73]
Predictions  [1 2 4 6 7 8]
Predictions map [45 48 68 70 71 73]

Result Analysis:
Total track-channel combinations: 14

Track structure in enhanced orchestration:
  Track 1: 3 Flutes (Ch 1) - 60 notes
  Track 1: 3 Flutes (Ch 2) - 60 notes
  Track 1: 3 Flutes (Ch 3) - 60 notes
  Track 2: 2 Oboes (Ch 4) - 66 notes
  Track 4: 2 Clarinets in A (Ch 6) - 53 notes
  Track 4: 2 Clarinets in A (Ch 7) - 53 notes
  Track 5: Bass Clarinet in Bb (Ch 2) - 53 notes
  Track 6: 2 Bassoons (Ch 8) - 4 notes
  Track 6: 2 Bassoons (Ch 10) - 4 notes
  Track 9: Violin I (Ch 13) - 121 notes
  Track 10: Violin II (Ch 14) - 121 notes
  Track 11: Viola (Ch 15) - 121 notes
  Track 12: Violoncello (Ch 0) - 122 notes
  Track 13: Contrabass (Ch 1) - 122 notes
Predictions  [1 2 4 6 7 8]
Predictions map [45 48 68 70 71 73]

Result Analysis:
Total track-channel combinations: 14

Track structure in enhanced